<a href="https://colab.research.google.com/github/gustavox0/Tecnicas-Avanzdas-ML/blob/main/M7_AG1_Grupo17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Contexto

IMDb (Internet Movie Database) es una base de datos de información de películas. Esta información incluye revisiones de las películas realizadas por los usuarios.
En esta práctica nos centraremos en analizar el sentimiento (positivo / negativo) de las revisiones.
Con todos estos datos, os pedimos lo siguiente:

# 1.Carga datasets

In [28]:
#Carga de librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Antes de comenzar vamos a revisar si tenemos disponible el entorno GPU

In [40]:
gpu_available = False
if len(tf.config.list_physical_devices('GPU')) > 0:
  gpu_available = True
  print("Running on GPU")

Running on GPU


In [50]:
# Carga del dataset

reviews_url = "https://raw.githubusercontent.com/md-lorente/Master_BD_DS/refs/heads/main/m%C3%B3dulo_7_aprendizaje_autom%C3%A1tico_para_machine_learning/reviews.txt"
labels_url = "https://raw.githubusercontent.com/md-lorente/Master_BD_DS/refs/heads/main/m%C3%B3dulo_7_aprendizaje_autom%C3%A1tico_para_machine_learning/labels.txt"

# Cargar archivos de texto sin encabezado
reviews = pd.read_csv(reviews_url, header=None, names=['review'])
labels = pd.read_csv(labels_url, header=None, names=['label'])

labels['label'] = labels['label'].apply(lambda x: 1 if x == 'positive' else 0)

In [51]:
# Muestra las primeras 5 reseñas y etiquetas
for i in range(5):
    print(f"Reseña #{i+1}: {reviews.iloc[i]['review']}")
    print(f"Etiqueta: {labels.iloc[i]['label']}")
    print("---")

Reseña #1: bromwell high is a cartoon comedy . it ran at the same time as some other programs about school life  such as  teachers  . my   years in the teaching profession lead me to believe that bromwell high  s satire is much closer to reality than is  teachers  . the scramble to survive financially  the insightful students who can see right through their pathetic teachers  pomp  the pettiness of the whole situation  all remind me of the schools i knew and their students . when i saw the episode in which a student repeatedly tried to burn down the school  i immediately recalled . . . . . . . . . at . . . . . . . . . . high . a classic line inspector i  m here to sack one of your teachers . student welcome to bromwell high . i expect that many adults of my age think that bromwell high is far fetched . what a pity that it isn  t   
Etiqueta: positive
---
Reseña #2: story of a man who has unnatural feelings for a pig . starts out with a opening scene that is a terrific example of absurd

In [52]:
#dimensiones de los datasets
print(f"Dimensiones del dataset de reseñas: {reviews.shape}")
print(f"Dimensiones del dataset de etiquetas: {labels.shape}")

Dimensiones del dataset de reseñas: (25000, 1)
Dimensiones del dataset de etiquetas: (25000, 1)


# 2.Preprocesado

## Eliminar signos que no sean alfanuméricos

In [55]:
#reemplazamos todo lo que no sea alfanumerico por un espacio y guardamos el texto limpio en reviews_clean
reviews_series = reviews.iloc[:, 0]
reviews_clean = []
for review in reviews_series:
    review = str(review)  # aseguramos que es string
    cleaned = re.sub(r"[^a-zA-Z0-9]", " ", review)
    reviews_clean.append(cleaned)

print(f"Total reviews_clean: {len(reviews_clean)}")
print(reviews_clean[:1]) #mostramos una reseña

Total reviews_clean: 25000
['bromwell high is a cartoon comedy   it ran at the same time as some other programs about school life  such as  teachers    my   years in the teaching profession lead me to believe that bromwell high  s satire is much closer to reality than is  teachers    the scramble to survive financially  the insightful students who can see right through their pathetic teachers  pomp  the pettiness of the whole situation  all remind me of the schools i knew and their students   when i saw the episode in which a student repeatedly tried to burn down the school  i immediately recalled                   at                     high   a classic line inspector i  m here to sack one of your teachers   student welcome to bromwell high   i expect that many adults of my age think that bromwell high is far fetched   what a pity that it isn  t   ']


## Tokenizar cada review

In [56]:
tokenized_reviews = [word_tokenize(review) for review in reviews_clean]

In [60]:
#mostrar 5 revies tokenizados y el numero de palabrtas
for i in range(5):
    print(f"Reseña #{i+1}: {tokenized_reviews[i]}")
    print(f"Número de palabras: {len(tokenized_reviews[i])}")
    print("---")

Reseña #1: ['bromwell', 'high', 'is', 'a', 'cartoon', 'comedy', 'it', 'ran', 'at', 'the', 'same', 'time', 'as', 'some', 'other', 'programs', 'about', 'school', 'life', 'such', 'as', 'teachers', 'my', 'years', 'in', 'the', 'teaching', 'profession', 'lead', 'me', 'to', 'believe', 'that', 'bromwell', 'high', 's', 'satire', 'is', 'much', 'closer', 'to', 'reality', 'than', 'is', 'teachers', 'the', 'scramble', 'to', 'survive', 'financially', 'the', 'insightful', 'students', 'who', 'can', 'see', 'right', 'through', 'their', 'pathetic', 'teachers', 'pomp', 'the', 'pettiness', 'of', 'the', 'whole', 'situation', 'all', 'remind', 'me', 'of', 'the', 'schools', 'i', 'knew', 'and', 'their', 'students', 'when', 'i', 'saw', 'the', 'episode', 'in', 'which', 'a', 'student', 'repeatedly', 'tried', 'to', 'burn', 'down', 'the', 'school', 'i', 'immediately', 'recalled', 'at', 'high', 'a', 'classic', 'line', 'inspector', 'i', 'm', 'here', 'to', 'sack', 'one', 'of', 'your', 'teachers', 'student', 'welcome', '

## Generar lista de palabras
 Generar lista de palabras distintas de todo el dataset y ordenar por número de veces que se presenta en el dataset, de forma que se puedan seleccionar las "20,000" más frecuentes

In [61]:
# Aplanar la lista de listas en una sola lista con todas las palabras
Palabras_dataset = [word.lower() for review in tokenized_reviews for word in review]

# Contar la frecuencia de cada palabra
Conteo_palabra = Counter(Palabras_dataset)

# Obtener las 20,000 palabras más frecuentes
Palabras_comunes = Conteo_palabra.most_common(20000)

# Mostrar las 10 primeras
print("Palabras más frecuentes:")
for palabra, count in Palabras_comunes[:10]:
    print(f"{palabra}: {count}")

Palabras más frecuentes:
the: 336713
and: 164107
a: 163009
of: 145864
to: 135720
is: 107328
br: 101872
it: 96352
in: 93968
i: 87623


## Identificar palabras de la lista

Asociar cada palabra de la lista con un número identificador

In [ ]:
palabra_to_index = {word: idx for idx, word in enumerate(Palabras_comunes)}

# Mostrar las primeras 5 palabras y su índice
for word, idx in list(palabra_to_index.items())[:5]:
    print(f"{word}: {idx}")

## Truncar reviews

Truncar las reviews a 160 palabras y rellener con "0" las reviews que tengas menos palabras

In [ ]:
#Convertir las reviews tokenizadas a secuencias de indices
reviews_seq = []

for tokens in tokenized_reviews:
    seq = [word_to_index[word] for word in tokens if word in word_to_index]
    reviews_seq.append(seq)

#truncar/rellenar las secuencias a longitud de 160
max_len = 160
reviews_trunc = pad_sequences(reviews_seq, maxlen=max_len, padding='post', truncating='post')
#con 'post' agregamos 0s a las reviews menores a 160 y con post, truncamos a 160 las reviews más largas.

print("Shape final:", reviews_trunc.shape)  # debería ser (25000, 160)
print("Ejemplo de reseña procesada:", reviews_trunc[0])

## Dividir datasets

Dividir los datasets de reviews y etiquetas en 80% para entrenamiento, 10% para validación y 10% para prueba.

In [ ]:
# División inicial: 80% entrenamiento, 20% temporal (validación + prueba)
X_train, X_temp, y_train, y_temp = train_test_split(
    reviews_trunc, labels, test_size=0.2, random_state=42)

# División de los datos temporales en 50% validación y 50% prueba => 10% y 10%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

print("Entrenamiento:", X_train.shape, y_train.shape)
print("Validación:", X_val.shape, y_val.shape)
print("Prueba:", X_test.shape, y_test.shape)

# 3.Carga fichero

Carga del fichero de palabras vectorizadas GloVe (https://materials.il3.ub.edu/cursos/tec_mbdds_taml/glove.6B/glove.6B.zip) de 100 dimensiones (se puede eliminar el resto de ficheros)

In [ ]:
# Cargamos el fichero de palabras vectorizadas de 100 dimensiones
from time import time
t_ini = time()

!rm -f glove.6B.zip
!wget https://materials.il3.ub.edu/cursos/tec_mbdds_taml/glove.6B/glove.6B.zip
!unzip -j glove.6B.zip "glove.6B.100d.txt" -d ./

print(f"Tiempo total: {time() - t_ini:.2f} segundos")

In [ ]:
# 5 primeras lineas del fichero
with open("glove.6B.100d.txt", "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()
        print(line.strip())

#4.Creacion del modelo 1

Definición de una Red Neuronal Recurrente para analizar el sentimiento de las reviews de películas del dataset.

    - El primer nivel será el embedding, cuya dimensión de salida será igual a la de los embeddings del fichero Glove. Estos embeddings serán entrenables: (trainable=True)
    - Añadir una capa LSM
    - Añadir una capa lineal

#5.Entrenamiento del modelo 1

Entrenamiento del modelo 1 y ver el impacto en *loss* y *accuracy*

# 6.Creación del modelo 2
Copiar la arquitectura del modelo 1 y añadir los siguientes cambios:
    - Añadir otro nivel a la red LSTM (parámetro n_layers).
    - Añadir bidireccionalidad.
    - Añadir capas de dropout.
    - Añadir otro nivel lineal
    - Sustituir los pesos de la capa de embeddings por los del fichero GloVe.
    - Con estas mejoras, intentad conseguir un *accuracy* superior al 80%.

# 7.Entrenamiento del modelo 2

Entrenamiento del modelo 2 y ver el impacto en loss y accuracy